<a href="https://colab.research.google.com/github/nivethithanm/mini-claw/blob/main/CLAW_03_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLAW-03 — Memory: Persistence Across Sessions

**Goal:** Your agent should remember you. Build a three-layer memory system.

By the end of this notebook you'll have:
- **L1 Short-term memory** — in-session conversation history (NB-01)
- **L2 Working memory** — key-value facts, persisted to disk as JSON
- **L3 Episodic memory** — past conversation summaries, searchable by recency
- A `MemoryManager` that manages all three layers
- Memory injection into the system prompt

> **First principles mindset:**  
> Memory = the agent's ability to be *different* after an interaction.  
> Without memory, every conversation starts from zero.  
> OpenClaw's "remembers you" magic is just smart memory injection.


## 0. Setup

In [1]:
import os, json, datetime, hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Any, Optional
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# Memory storage location — mimics OpenClaw's ~/.claw/ directory
MEMORY_DIR = Path("./claw_memory")
MEMORY_DIR.mkdir(exist_ok=True)
(MEMORY_DIR / "episodes").mkdir(exist_ok=True)

print(f"Memory dir: {MEMORY_DIR.resolve()}")


Memory dir: /content/claw_memory


## 1. L2 — Working Memory (Key-Value Facts)

Working memory holds structured facts: user preferences, important context,
entity information. It's small, always loaded, and injected into the system prompt.

```
{
    "user_name": "Nivin",
    "preferred_language": "Python",
    "location": "Chennai",
    "communication_style": "concise, technical",
}
```


In [2]:
class WorkingMemory:
    """
    Key-value store for persistent facts.

    Backed by a JSON file so it survives restarts.
    Injected into the system prompt as a context block.
    """

    def __init__(self, path: Path = MEMORY_DIR / "working_memory.json"):
        self.path = path
        self._data: dict[str, str] = {}
        self.load()

    def load(self):
        """Load from disk if the file exists."""
        # TODO: if self.path exists, load JSON into self._data
        if self.path.exists():
            with open(self.path) as f:
                self._data = json.load(f)
        else:
            self._data = {}

    def save(self):
        """Persist to disk."""
        # TODO: write self._data to self.path as JSON (indent=2)
        with open(self.path, "w") as f:
            json.dump(self._data, f, indent=2)

    def set(self, key: str, value: str):
        """Store a fact and immediately persist."""
        # TODO
        self._data[key] = value
        self.save()

    def get(self, key: str, default: str = None) -> Optional[str]:
        return self._data[key]

    def delete(self, key: str):
        self._data.pop(key, None)
        self.save()

    def as_prompt_block(self) -> str:
        """
        Format memory as a readable block for the system prompt.

        Output example:
        ## What I know about you:
        - user_name: Niv
        - location: Chennai
        """
        # TODO: return empty string if no data, else format as above
        if not self._data:
            return ""
        lines = ["## What I know about you:"]
        for k, v in self._data.items():
            lines.append(f"- {k}: {v}")
        return "\n".join(lines)

    def __repr__(self):
        return f"WorkingMemory({len(self._data)} entries)"

In [3]:
# Test
wm = WorkingMemory()
wm.set("user_name", "Niv")
wm.set("preferred_language", "Python")
wm.set("communication_style", "concise and technical")

print(wm)
print(wm.as_prompt_block())
print()
# Reload from disk to verify persistence
wm2 = WorkingMemory()
assert wm2.get("user_name") == "Niv"
print("WorkingMemory persistence test pass ✓")
print("Prompt block: ", wm.as_prompt_block())


WorkingMemory(3 entries)
## What I know about you:
- user_name: Niv
- preferred_language: Python
- communication_style: concise and technical

WorkingMemory persistence test pass ✓
Prompt block:  ## What I know about you:
- user_name: Niv
- preferred_language: Python
- communication_style: concise and technical


## 2. L3 — Episodic Memory (Conversation Summaries)

Episodic memory stores *what happened* in past sessions, as LLM-generated summaries.

Each episode has:
- Timestamp
- A 3-5 sentence summary generated by the LLM
- Key entities/facts extracted from the conversation
- A unique ID

When starting a new session, the agent loads the 3 most recent episodes to
remember recent context.


In [4]:
@dataclass
class Episode:
    id: str
    timestamp: str
    summary: str
    key_facts: list[str]
    turn_count: int

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "Episode":
        return cls(**d)

    def as_prompt_block(self) -> str:
        facts_str = "\n".join(f"  - {f}" for f in self.key_facts)
        return f"[{self.timestamp}] {self.summary}\nKey facts:\n{facts_str}"


def summarize_conversation(messages: list[dict], client: OpenAI) -> tuple[str, list[str]]:
    """
    Ask the LLM to summarize a conversation.

    Returns: (summary_text, list_of_key_facts)

    Exercise: craft a prompt that asks the LLM to produce:
    1. A 3-sentence summary
    2. A JSON list of key facts to remember

    Hint: ask for JSON output with keys "summary" and "key_facts"
    """
    # TODO
    convo_text = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in messages
        if m["role"] in ("user", "assistant") and m.get("content")
    )

    prompt = '''
    Summarize the following conversation in 3 sentences.
    Then extract up to 5 key facts worth remembering long-term (user preferences, decisions, important context).

    Respond ONLY with valid JSON in this exact format:
    {{
      "summary": "...",
      "key_facts": ["fact 1", "fact 2", ...]
    }}

    Conversation:
    {convo_text}
    '''.format(convo_text=convo_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    # Strip JSON code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        data = json.loads(raw)
        return data["summary"], data.get("key_facts", [])
    except Exception:
        return raw, []

In [5]:
# Test with a sample conversation
sample_messages = [
    {"role": "user", "content": "Hi! I'm Niv, an ML engineer from Chennai."},
    {"role": "assistant", "content": "Great to meet you, Niv! What are you working on?"},
    {"role": "user", "content": "I'm building a personal AI agent. I prefer Python and want it to connect to Telegram."},
    {"role": "assistant", "content": "Interesting! Python + Telegram is a great combo. I'll help you build it."},
]

summary, facts = summarize_conversation(sample_messages, client)
print("Summary:", summary)
print("Key facts:", facts)

Summary: Niv, an ML engineer from Chennai, is working on building a personal AI agent. He prefers using Python for the project and wants it to connect to Telegram. The assistant expresses enthusiasm and offers to help him with the development.
Key facts: ["User's name is Niv", 'Niv is an ML engineer from Chennai', 'Niv prefers Python for his project', 'Niv wants his AI agent to connect to Telegram', 'The assistant is willing to help Niv build the AI agent']


In [7]:
class EpisodicMemory:
    """
    Stores conversation summaries on disk.
    One JSON file per episode, named by timestamp.
    """
    def __init__(self, episodes_dir: Path = MEMORY_DIR / "episodes"):
        self.episodes_dir = episodes_dir
        self.episodes_dir.mkdir(exist_ok=True)

    def save_episode(self, messages: list[dict], turn_count: int) -> Episode:
        """
        Summarize a conversation and save it as an episode.

        Exercise: call summarize_conversation, create an Episode, save to disk.
        Use timestamp as the filename: YYYYMMDD_HHMMSS.json
        """
        summary, key_facts = summarize_conversation(messages, client)
        timestamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
        ep_id = hashlib.md5(timestamp.encode()).hexdigest()[:8]
        episode = Episode(
            id=ep_id,
            timestamp=timestamp,
            summary=summary,
            key_facts=key_facts,
            turn_count=turn_count,
        )
        path = self.episodes_dir / f"{timestamp}.json"
        with open(path, "w") as f:
            json.dump(episode.to_dict(), f, indent=2)
        return episode

    def load_recent(self, n: int = 3) -> list[Episode]:
        """
        Load the n most recent episodes, sorted newest-first.

        Exercise: list all JSON files in episodes_dir, sort by name (which is timestamp),
        load the last n, return as Episode objects.
        """
        files = sorted(self.episodes_dir.glob("*.json"), reverse=True)[:n]
        episodes = []
        for f in files:
            with open(f) as fp:
                episodes.append(Episode.from_dict(json.load(fp)))
        return episodes

    def as_prompt_block(self, n: int = 3) -> str:
        """
        Format recent episodes as a readable block for the system prompt.
        """
        episodes = self.load_recent(n)
        if not episodes:
            return ""
        lines = ["## Recent session history:"]
        for ep in episodes:
            lines.append(ep.as_prompt_block())
            lines.append("")
        return "\n".join(lines)

    def count(self) -> int:
        return len(list(self.episodes_dir.glob("*.json")))


In [8]:
# Test
em = EpisodicMemory()
ep = em.save_episode(sample_messages, turn_count=2)
print(f"Saved episode: {ep.id}")
print()
recent = em.load_recent(1)
print("Most recent episode:")
print(recent[0].as_prompt_block())

Saved episode: 50599167

Most recent episode:
[20260607_092913] Niv, an ML engineer from Chennai, is working on building a personal AI agent. He prefers using Python for the project and wants it to connect to Telegram. The assistant expresses enthusiasm and offers to help Niv with his project.
Key facts:
  - User's name is Niv
  - Niv is an ML engineer from Chennai
  - Niv prefers Python for his AI agent
  - Niv wants the AI agent to connect to Telegram
  - The assistant is willing to help Niv with his project


/tmp/ipykernel_1195/4127823946.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


## 3. MemoryManager — Unified Interface

The agent only talks to `MemoryManager`. It handles all three layers.


In [9]:
class MemoryManager:
    """
    Unified interface to all memory layers.

    The agent loop will:
    1. Call inject_into_system_prompt() at the start of each session
    2. Call set_fact() when the LLM identifies something to remember
    3. Call save_session() at the end of each session
    """

    def __init__(self, memory_dir: Path = MEMORY_DIR):
        self.working = WorkingMemory(memory_dir / "working_memory.json")
        self.episodic = EpisodicMemory(memory_dir / "episodes")

    def set_fact(self, key: str, value: str):
        """Store a long-term fact."""
        self.working.set(key, value)

    def get_fact(self, key: str) -> Optional[str]:
        return self.working.get(key)

    def inject_into_system_prompt(self, base_prompt: str) -> str:
        """
        Augment a base system prompt with memory context.

        Exercise: concatenate:
        1. base_prompt
        2. working memory block (if any)
        3. episodic memory block (if any)
        """
        parts = [base_prompt]
        wm_block = self.working.as_prompt_block()
        if wm_block:
            parts.append("\n\n" + wm_block)
        ep_block = self.episodic.as_prompt_block(n=3)
        if ep_block:
            parts.append("\n\n" + ep_block)
        return "".join(parts)

    def save_session(self, messages: list[dict], turn_count: int):
        """
        Summarize and archive the current session.
        Only save if turn_count > 1 (skip trivial sessions).
        """
        if turn_count <= 1:
            return
        self.episodic.save_episode(messages, turn_count)

    def stats(self):
        print(f"Working memory: {len(self.working._data)} facts")
        print(f"Episodes: {self.episodic.count()}")


In [10]:
# Test full memory manager
mm = MemoryManager()
BASE_PROMPT = "You are Claw, a personal AI assistant."
full_prompt = mm.inject_into_system_prompt(BASE_PROMPT)
print("Augmented system prompt:")
print(full_prompt)
print()
mm.stats()

Augmented system prompt:
You are Claw, a personal AI assistant.

## What I know about you:
- user_name: Niv
- preferred_language: Python
- communication_style: concise and technical

## Recent session history:
[20260607_092913] Niv, an ML engineer from Chennai, is working on building a personal AI agent. He prefers using Python for the project and wants it to connect to Telegram. The assistant expresses enthusiasm and offers to help Niv with his project.
Key facts:
  - User's name is Niv
  - Niv is an ML engineer from Chennai
  - Niv prefers Python for his AI agent
  - Niv wants the AI agent to connect to Telegram
  - The assistant is willing to help Niv with his project


Working memory: 3 facts
Episodes: 1


## 4. Memory-Aware Tools

Add two tools that let the LLM explicitly manage memory.


In [11]:
def make_memory_tools(mm: MemoryManager):
    """Build remember/recall tools backed by the MemoryManager."""
    from dataclasses import dataclass, field

    def _remember(key: str, value: str) -> str:
        mm.set_fact(key, value)
        return f"Remembered: {key} = {value}"

    def _recall(key: str) -> str:
        val = mm.get_fact(key)
        return val if val else f"No memory found for '{key}'"

    def _list_memory() -> str:
        facts = mm.working._data
        if not facts:
            return "No facts stored yet."
        return "\n".join(f"{k}: {v}" for k, v in facts.items())

    # Using simple dicts instead of dataclass for brevity here
    # (defined inline for notebook self-containment)

    class SimpleTool:
        def __init__(self, name, description, params, required, fn):
            self.name = name
            self.description = description
            self.parameters = params
            self.required = required
            self.fn = fn
        def to_openai_spec(self):
            return {"type": "function", "function": {
                "name": self.name, "description": self.description,
                "parameters": {"type": "object", "properties": self.parameters, "required": self.required}
            }}
        def execute(self, **kwargs):
            return str(self.fn(**kwargs))

    return [
        SimpleTool("remember", "Store a fact long-term.",
            {"key": {"type": "string"}, "value": {"type": "string"}}, ["key", "value"], _remember),
        SimpleTool("recall", "Retrieve a stored fact.",
            {"key": {"type": "string"}}, ["key"], _recall),
        SimpleTool("list_memory", "List all stored facts.", {}, [], _list_memory),
    ]

# Note: In NB-06 you'll wire these properly into the ToolRegistry from NB-02
print("Memory tools factory defined ✓")


Memory tools factory defined ✓


## 5. Memory-Injected Agent

Combine memory with your NB-01 chat session for a fully persistent agent.


In [12]:
from openai import OpenAI

class MemoryAgent:
    """
    A ChatSession + MemoryManager.

    At start: injects memory into system prompt.
    During session: agent can call remember/recall tools.
    At end: saves episode summary.
    """

    def __init__(
        self,
        base_system_prompt: str,
        memory_dir: Path = MEMORY_DIR,
        model: str = "gpt-4o-mini",
    ):
        self.base_prompt = base_system_prompt
        self.memory = MemoryManager(memory_dir)
        self.model = model
        self.history: list[dict] = []  # OpenAI format dicts
        self._turn_count = 0

        # Build augmented system prompt
        self.system_prompt = self.memory.inject_into_system_prompt(base_system_prompt)
        print(f"System prompt tokens (approx): {len(self.system_prompt.split())}")

    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        messages = [{"role": "system", "content": self.system_prompt}] + self.history

        response = client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        self._turn_count += 1
        return reply

    def end_session(self):
        """Call this when the user closes the session — saves episode."""
        self.memory.save_session(self.history, self._turn_count)
        print(f"Session saved. {self._turn_count} turns archived.")

    def __repr__(self):
        return f"MemoryAgent(turns={self._turn_count}, model={self.model})"


# Demo
agent = MemoryAgent(
    base_system_prompt="You are Claw. You are helpful and you remember everything about the user.",
)

r1 = agent.chat("My name is Niv and I'm building an AI agent framework.")
print(f"Agent: {r1}\n")

r2 = agent.chat("I prefer minimal, readable Python code over complex abstractions.")
print(f"Agent: {r2}\n")

# End session — saves summary
agent.end_session()

# New session — starts with the old episode in the prompt
print("\n--- Starting new session ---")
agent2 = MemoryAgent(
    base_system_prompt="You are Claw. You remember everything about the user.",
)
r3 = agent2.chat("Do you remember what I'm working on?")
print(f"Agent: {r3}")


System prompt tokens (approx): 119
Agent: Got it, Niv! You're building an AI agent framework. How can I assist you with that?

Agent: Understood, Niv. I'll focus on providing minimal and readable Python code for your AI agent framework. What specific functionality or feature would you like to implement next?



/tmp/ipykernel_1195/4127823946.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


Session saved. 2 turns archived.

--- Starting new session ---
System prompt tokens (approx): 198
Agent: Yes, you're building a personal AI agent framework in Python, and you want it to connect to Telegram. If you need help with any specific functionality or feature, just let me know!


## 6. Exercises

**E1.** Add a `forget(key)` tool that removes a fact from working memory and tells the user it's been forgotten.

**E2.** Implement **memory deduplication**: before saving a fact, check if a very similar key already exists (use string similarity or ask the LLM). Avoid storing `user_name` and `name` as separate entries.

**E3.** Add an `EpisodicMemory.search(query)` method that uses OpenAI embeddings to find relevant past episodes (not just the most recent). This is the RAG version of episodic memory.

**E4 (hard):** Implement **automatic fact extraction** — after every assistant response, run a background LLM call (non-blocking) that reads the last 2 turns and decides if any new long-term facts should be stored. This is how OpenClaw's memory "feels magic."

---

## ✅ Checkpoint

You now have:
- `WorkingMemory`: persistent key-value facts
- `EpisodicMemory`: session summaries with key facts
- `MemoryManager`: unified interface + prompt injection
- `MemoryAgent`: session that remembers across restarts

**Next:** CLAW-04 — Skill System. Your agent gets extensible superpowers.
